**Cell 1**

# Role 3 - Preprocessing and signal validation

**Owner: unassigned**

Own the complete preprocessing chain: PSD, whitening, bandpass, crop, and numerical/physical checks. This is the dedicated preprocessing role. You do not acquire production data, create labels or train models. Your checkpoint deliverable is data evidence, not a trained model.

Revised scope: dataset preparation, preprocessing and EDA. This is a new notebook; earlier role notebooks are preserved. Every implementation cell is deliberately empty.

**Cell 2**

## What this milestone covers

The project question remains whether four-parameter inference generalizes from O3a H1 noise to O4a H1 noise. This checkpoint concentrates on preparing and understanding the dataset: source provenance, physical labels, preprocessing, actual split counts, EDA, plots, and limitations. Model training, model performance tables and calibration conclusions are outside this notebook set's required work.

The methodology section must still describe three algorithms because the rubric asks for a baseline and two improvements: linear regression, its small regularized extension Ridge, and SNPE with a masked autoregressive flow. Describe them as the next common comparison; do not invent new training results. Feature importance is explicitly requested by the rubric. Input-target association is useful EDA but does not fully substitute for model-based importance; Role 4 must state this gap honestly if importance is deferred.

**Existing evidence versus your work:** The repository already contains saved noise, injections and processed data. The revised PDF uses those artifacts, not results from these new notebooks. Your task is a separately identified manual pilot. Do not claim your pilot has the old dataset's 800-per-run counts until you actually generate and check them. The earlier split was a 750-row O3a development pool, 50 O3a held-out examples and 800 O4a evaluation examples. A 600/150 subdivision of the development pool is a proposal, not a completed split.

**Implementation rule:** Write your own code in the empty cells. You may call standard scientific libraries and consult their documentation, but do not import ligo_pipeline, project utils, config.py, another notebook's implementation, or prepared training/model checkpoints. Trusted raw strain with source metadata is allowed. You may write your own small functions after explaining their steps. No solution code is supplied here.

**Ownership:** Assign one person per notebook; names are unassigned. Review each other's outputs, but let each owner make their own notebook changes. Record actual contributions and dates. Every cell has a sequential cell_number and every markdown cell displays its number; the preceding task labels the empty code cell. Renumber all cells from 1 when inserting new cells. Execution counters are not stable cell numbering.


**Cell 3**

## Common contract and parallel work

Agree on one small experiment before scaling up: H1; O3a development and within-run evaluation; O4a cross-run evaluation; 4096 Hz; four-second final windows; a 35-350 Hz preprocessing band; target order chirp_mass, mass_ratio, chi_eff, distance. Keep any fixed-mass, distance-only exercise separate from this four-target pilot.

Use a new experiment_id and store files under artifacts/manual_preprocessing_milestone/<experiment_id>/. The role-owned subfolders are data/, simulation/, preprocessing/ and analysis/. Put experiment.json in data/ and repeat its ID and relevant settings in each output's metadata. Store raw segment duration, sample rate, waveform assumptions, parameter ranges, seeds, intended counts, actual counts and units. Do not overwrite the existing artifacts or the older role notebooks.

Role 1 supplies raw arrays, noise_id/group_id, source GPS intervals and source split assignments. Role 3 supplies noise-only PSD files early. Role 2 uses those PSDs for SNR selection, then supplies padded injected buffers, merger indices and labels. Role 3 supplies processed arrays and row metadata. Role 4 audits and combines evidence, counts and EDA; this role does not train the final models.

**Source-group separation:** Assign groups before simulation. Overlapping raw GPS intervals are one group even if filenames differ. Use train, validation, O3a_test and O4a_test split names; keep all O4a out of fitting and tuning. A recommended small design has at least three O3a training groups, one O3a validation group, one O3a test group and two O4a groups. Choose feasible durations and target counts together; these are planning suggestions, not completed allocations. If your pilot cannot support this design, document the limitation. Track nonoverlapping final crop windows as well as group identities.

**Handoff format:** Use .npy/.npz numeric arrays and .csv/.json metadata with repo-relative paths. Join per-example records by unique sample_id; join noise/PSD records by noise_id. Preserve run, split, group and units. Never assume two files have matching row order. Save rejection ledgers, shapes, dtypes, versions and seeds. Changes to schema/settings require all consumers to check compatibility.

**Start in parallel:** Role 1 can inspect one raw segment, Role 2 can inspect clean waveforms, Role 3 can validate sinusoids/FFT/filtering, and Role 4 can design the counting ledger, plot checklist and split audit. First integrate one real example end to end through data preparation. Then expand the pilot; no model fit is needed to check these handoffs.


**Cell 4**

## Your work boundary

**Inputs:** Role 1 raw noise and source metadata; Role 2 padded injected buffers with merger indices and target IDs. Until those arrive, use simple sinusoids/noise and one real raw segment to learn the transformations.

**Outputs:** preprocessing/psd/<noise_id>.npz; preprocessing/processed_manifest.csv; preprocessing/processed.npz; preprocessing/rejections.csv; before/after plots and processing_settings.json. Deliver PSD files early so Role 2 can calculate SNR.

**Start in parallel:** You can study FFTs, validate frequency axes, design a filter and implement crop boundary checks immediately. Export a noise PSD as soon as Role 1 supplies one segment. Finish real whitening/cropping once Role 2 supplies one padded injection. The full dataset is not needed to prove your methods.

**File contract:** Each PSD .npz contains frequency_hz and psd_strain2_per_hz and is identified by noise_id; its metadata records source and Welch settings. processed.npz contains X with shape (N,16384), theta with shape (N,4), sample_id and split arrays. processed_manifest.csv adds row_index, sample_id, noise_id, group_id, run, split, n_samples, finite_status and source buffer path. processed arrays and manifests must agree exactly; use explicit numeric/string dtypes, not pickled objects.

All paths use the experiment-specific root in the common contract above. Older path wording in conceptual examples does not authorize writing into historical artifact folders.

**Cell 5**

## Task 1: Build the time and frequency foundations

### Background knowledge
FFT bins are fk = k*fs/N, the Nyquist frequency is fs/2, and frequency resolution is fs/N. A raw FFT magnitude is not a PSD. For real inputs, a one-sided transform avoids duplicate negative-frequency bins. Windowing changes spectral leakage.

### Your task
Construct a known sinusoid with a duration and sample rate you choose. Compute its frequency axis and show where its peak should be. Compare windowed/unwindowed spectra, then inspect one real segment. Explain bin spacing, Nyquist and the 35-350 Hz analysis band.

### Evidence and completion checks
Time/spectrum plots and a measured sinusoid peak error; axes include seconds, Hz and units.

**Cell 6 is your empty work cell.** Write your own implementation if this task needs code. For planning/writing-only tasks, leave it empty and complete your response below. Do not train a model merely to fill a cell.

**Cell 7**

### Your findings and contribution record

- What did you do personally, and on what date?
- What measured counts, plots or checks support your answer?
- What are the units, IDs and assumptions?
- What failed, remains planned or needs a teammate's input?
- What will you hand off?

**Your response:**

**Cell 8**

## Task 2: Estimate a noise-only PSD and export it early

### Background knowledge
Welch PSD averages overlapping windowed periodograms and has units strain squared per Hz. ASD is its square root. More averaging reduces variability at the expense of frequency resolution. Estimate background PSD from the source noise, not from a buffer containing the injected signal.

### Your task
Implement and inspect a small Welch-style calculation, then compare with a trusted library result. Use four-second subwindows when the raw segment is long enough; record overlap, window and detrending. Check increasing frequency, nonnegative PSD and units. Explain a positive floor for unusable/near-zero bins. Export the first PSD and its settings to Role 2.

### Evidence and completion checks
A trustworthy PSD artifact, PSD/ASD plot, normalization check and documented floor/invalid-bin policy.

**Cell 9 is your empty work cell.** Write your own implementation if this task needs code. For planning/writing-only tasks, leave it empty and complete your response below. Do not train a model merely to fill a cell.

**Cell 10**

### Your findings and contribution record

- What did you do personally, and on what date?
- What measured counts, plots or checks support your answer?
- What are the units, IDs and assumptions?
- What failed, remains planned or needs a teammate's input?
- What will you hand off?

**Your response:**

**Cell 11**

## Task 3: Whiten without erasing physical signal

### Background knowledge
Whitening divides frequency components by the square root of a matching PSD, then transforms back. Constants depend on one-/two-sided PSD and FFT conventions. It is not feature-wise StandardScaler normalization, and an arbitrary unit-variance result alone does not prove correctness.

### Your task
Choose and explain a normalization convention. Interpolate the noise PSD to the buffer FFT grid and apply whitening yourself. Compare spectra before and after on a noise-only interval. Check finite values and reconstruction behavior. Preserve the PSD scale convention for all runs; explain any floor or zero-bin handling.

### Evidence and completion checks
A derivation in your words, before/after PSD and time plots, and empirical checks of frequency flattening.

**Cell 12 is your empty work cell.** Write your own implementation if this task needs code. For planning/writing-only tasks, leave it empty and complete your response below. Do not train a model merely to fill a cell.

**Cell 13**

### Your findings and contribution record

- What did you do personally, and on what date?
- What measured counts, plots or checks support your answer?
- What are the units, IDs and assumptions?
- What failed, remains planned or needs a teammate's input?
- What will you hand off?

**Your response:**

**Cell 14**

## Task 4: Apply and validate the bandpass

### Background knowledge
A bandpass removes frequencies outside the analysis band. Forward/backward filtering reduces phase delay but has boundary effects; the effective response is applied twice. Second-order sections can improve numerical stability. Zero-phase filtering is an offline operation.

### Your task
Design a fourth-order Butterworth 35-350 Hz bandpass for fs=4096 Hz. Plot its response. Test a below-band, in-band and above-band sinusoid and measure attenuation. Inspect both edges and decide how much padding is needed. Record design order and forward/backward behavior.

### Evidence and completion checks
Response plot, three-sinusoid check, edge inspection and exact settings.

**Cell 15 is your empty work cell.** Write your own implementation if this task needs code. For planning/writing-only tasks, leave it empty and complete your response below. Do not train a model merely to fill a cell.

**Cell 16**

### Your findings and contribution record

- What did you do personally, and on what date?
- What measured counts, plots or checks support your answer?
- What are the units, IDs and assumptions?
- What failed, remains planned or needs a teammate's input?
- What will you hand off?

**Your response:**

**Cell 17**

## Task 5: Crop only after processing padded buffers

### Background knowledge
The project convention is two seconds before plus two seconds after merger: four seconds total, or 16384 samples at 4096 Hz. Cropping first can create artificial filter/FFT boundaries near the signal. Local buffer indices differ from GPS/source indices.

### Your task
Use Role 2 local merger indices to crop after whitening and filtering. Verify the peak alignment on one known injection. Check exact length, finiteness and enough padding for your documented edge policy. Reject invalid bounds rather than silently truncate. Explain what happens for a near-edge candidate.

### Evidence and completion checks
One final input plot with merger marker, sample-count proof and a rejection ledger.

**Cell 18 is your empty work cell.** Write your own implementation if this task needs code. For planning/writing-only tasks, leave it empty and complete your response below. Do not train a model merely to fill a cell.

**Cell 19**

### Your findings and contribution record

- What did you do personally, and on what date?
- What measured counts, plots or checks support your answer?
- What are the units, IDs and assumptions?
- What failed, remains planned or needs a teammate's input?
- What will you hand off?

**Your response:**

**Cell 20**

## Task 6: Assemble data without changing labels or splits

### Background knowledge
A target row must stay paired with the same observation through all filtering and reordering. Training-only ML scaling belongs to the model workflow; per-segment background PSD estimation is signal processing and must use a consistent inference-time policy.

### Your task
Join buffers and labels by sample_id. Preserve group and split. Export X, theta and ID/split arrays plus a row manifest. Audit every join and give actual per-split counts. Do not add feature scaling fitted across all rows or normalize each waveform amplitude casually; amplitude helps infer distance.

### Evidence and completion checks
Role 4 can load fixed-shape inputs and labels and verify ID order without using your functions.

**Cell 21 is your empty work cell.** Write your own implementation if this task needs code. For planning/writing-only tasks, leave it empty and complete your response below. Do not train a model merely to fill a cell.

**Cell 22**

### Your findings and contribution record

- What did you do personally, and on what date?
- What measured counts, plots or checks support your answer?
- What are the units, IDs and assumptions?
- What failed, remains planned or needs a teammate's input?
- What will you hand off?

**Your response:**

**Cell 23**

## Task 7: Explain the evidence for the milestone

### Background knowledge
A processing description must distinguish invalid samples, valid loud observations, normalization, feature selection and categorical encoding. It should explain how the transformations affect the scientific signal.

### Your task
Build one multipanel raw/whitened/filtered/cropped figure, with units and matching time axes where possible. Write 150 words covering missing values, outliers, whitening, filtering, cropping and numerical issues. State no categorical encoding if inputs are numeric, and no learned feature selection if none was performed. Keep your contribution log.

### Evidence and completion checks
A self-contained preprocessing section, labeled figure, settings file and explicit completed/planned status.

**Cell 24 is your empty work cell.** Write your own implementation if this task needs code. For planning/writing-only tasks, leave it empty and complete your response below. Do not train a model merely to fill a cell.

**Cell 25**

### Your findings and contribution record

- What did you do personally, and on what date?
- What measured counts, plots or checks support your answer?
- What are the units, IDs and assumptions?
- What failed, remains planned or needs a teammate's input?
- What will you hand off?

**Your response:**

**Cell 26**

## Milestone completion and handoff

Before marking your role complete, show the teammate consuming your artifacts one real record and its provenance. Give them a settings file, retained/rejected counts and a short interpretation, not just an array. Check sample_id, noise_id, group_id, run, split and shape at the handoff. Save outputs under your role-owned folder for the same experiment_id.

Deliver your completed notebook, a contribution log, 100-150 words of findings, figures with descriptive titles/axes/units/legends/captions and an explicit list of unfinished checks. Do not copy another person's implementation into your notebook. Preserve the older notebooks and existing PDFs.

**Stop at this checkpoint:** dataset preparation, validated preprocessing and EDA, exact allocation tables and the three-algorithm methodology plan. Full linear/Ridge/SNPE training, final model ranking, posterior calibration results and 800-per-run manual regeneration are not mandatory tasks in this set. Feature importance remains an acknowledged rubric gap if deferred; the team must decide how to complete that requirement before submission rather than asserting full compliance.

**Write a final reflection:** What did you personally implement? What evidence supports it? Which decisions need another owner's agreement? Which findings belong to the new pilot rather than the historical project? What remains before submission?


**Cell 27**

## Background references

- [GWOSC data](https://gwosc.org/data/)
- [PyCBC waveforms](https://pycbc.org/pycbc/latest/html/waveform.html)
- [SciPy signal processing](https://docs.scipy.org/doc/scipy/reference/signal.html)
- [sbi calibration guide](https://sbi.readthedocs.io/en/latest/how_to_guide/16_sbc.html)

Consult library documentation for concepts and conventions. Your notebook should contain your own implementation and interpretation.